# Частина 2. Аналіз датасету електроспоживання
## Завдання 1: Завантаження та очищення даних від пропусків

In [1]:
import pandas as pd
import numpy as np
import timeit

# Завантажуємо дані. Оскільки файл великий, це може зайняти до 10-15 секунд.
# Пропуски в цьому датасеті позначені знаком "?"
df_power = pd.read_csv('household_power_consumption.txt', sep=';', 
                       low_memory=False, na_values=['?'])

# Видаляємо рядки з пропусками
df_power = df_power.dropna()

# Конвертуємо числові стовпці у правильний формат float
numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
                'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
df_power[numeric_cols] = df_power[numeric_cols].astype(float)

df_power.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


## Завдання 2: Вибірка 1 (Активна потужність > 5 кВт) з профілюванням часу

In [2]:
def filter_active_power(dataframe):
    return dataframe[dataframe['Global_active_power'] > 5.0]

# Профілювання часу виконання за допомогою timeit
execution_time = timeit.timeit(lambda: filter_active_power(df_power), number=10)
print(f"Середній час виконання функції (10 запусків): {execution_time / 10:.5f} секунд")

filter_active_power(df_power).head()

Середній час виконання функції (10 запусків): 0.01462 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0


## Завдання 3: Вибірка 2 (Сила струму 19-20 А, порівняння споживання побутових приладів)

In [3]:
def filter_intensity_and_appliances(dataframe):
    # Струм в межах 19-20 А
    cond1 = dataframe['Global_intensity'].between(19.0, 20.0)
    # Пральна машина (Sub_1) + Холодильник (Sub_2) > Бойлер/Кондиціонер (Sub_3)
    cond2 = (dataframe['Sub_metering_1'] + dataframe['Sub_metering_2']) > dataframe['Sub_metering_3']
    
    return dataframe[cond1 & cond2]

execution_time = timeit.timeit(lambda: filter_intensity_and_appliances(df_power), number=10)
print(f"Час виконання: {execution_time / 10:.5f} сек")

filter_intensity_and_appliances(df_power).head()

Час виконання: 0.04936 сек


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0


## Завдання 4: Вибірка 3 (Випадкові 500,000 записів та середнє для груп споживання)

In [4]:
def random_sample_stats(dataframe):
    sample_df = dataframe.sample(n=500000, replace=False)
    means = sample_df[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    return means

execution_time = timeit.timeit(lambda: random_sample_stats(df_power), number=5)
print(f"Час виконання: {execution_time / 5:.5f} сек")

random_sample_stats(df_power)

Час виконання: 0.42197 сек


Sub_metering_1    1.122026
Sub_metering_2    1.287962
Sub_metering_3    6.455132
dtype: float64

## Завдання 5: Нормалізація, Стандартизація та Коефіцієнти кореляції (Пірсона/Спірмена)

In [5]:
!pip install scipy


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# Нормалізація (Min-Max) для стовпця Voltage
df_power['Voltage_Normalized'] = (df_power['Voltage'] - df_power['Voltage'].min()) / (df_power['Voltage'].max() - df_power['Voltage'].min())

# Стандартизація (Z-score) для стовпця Global_active_power
df_power['Active_Power_Standardized'] = (df_power['Global_active_power'] - df_power['Global_active_power'].mean()) / df_power['Global_active_power'].std()

# Коефіцієнти кореляції між двома атрибутами
pearson_corr = df_power['Global_active_power'].corr(df_power['Global_intensity'], method='pearson')
spearman_corr = df_power['Global_active_power'].corr(df_power['Global_intensity'], method='spearman')

print(f"Коефіцієнт Пірсона: {pearson_corr}")
print(f"Коефіцієнт Спірмена: {spearman_corr}")

Коефіцієнт Пірсона: 0.9988886002095853
Коефіцієнт Спірмена: 0.9953719367299743


## Завдання 6: Складна вибірка (Час > 18:00, Потужність > 6 кВт, Група 2 — найбільша, вибірка за кроком)

In [7]:
def complex_time_filter(dataframe):
    # 1. Фільтруємо за часом після 18:00:00 та активною потужністю > 6 кВт
    cond_time_power = (dataframe['Time'] > '18:00:00') & (dataframe['Global_active_power'] > 6.0)
    filtered_df = dataframe[cond_time_power].copy()
    
    # 2. Визначаємо, де група 2 (пральна машина, холодильник тощо) є найбільшою серед усіх трьох груп
    cond_group2_max = (filtered_df['Sub_metering_2'] > filtered_df['Sub_metering_1']) & \
                      (filtered_df['Sub_metering_2'] > filtered_df['Sub_metering_3'])
    final_filtered = filtered_df[cond_group2_max]
    
    if final_filtered.empty:
        print("Записи, що відповідають усім критеріям, не знайдені.")
        return final_filtered

    # 3. Ділимо датафрейм навпіл
    half_len = len(final_filtered) // 2
    
    # Беремо кожен 3-й результат із першої половини та кожен 4-й із другої половини
    first_half_sliced = final_filtered.iloc[:half_len:3]
    second_half_sliced = final_filtered.iloc[half_len::4]
    
    # Об'єднуємо дві частини назад в один результат
    result = pd.concat([first_half_sliced, second_half_sliced])
    return result

# Профілювання часу виконання функції за допомогою timeit (5 запусків)
execution_time = timeit.timeit(lambda: complex_time_filter(df_power), number=5)
print(f"Середній час виконання складної вибірки: {execution_time / 5:.5f} секунд")

# Виводимо результат вибірки на екран
complex_time_filter(df_power).head()

Середній час виконання складної вибірки: 0.42191 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Voltage_Normalized,Active_Power_Standardized
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0,0.314378,4.691585
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0,0.292407,4.933712
17494,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0,0.433926,5.007485
17498,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0,0.397415,6.617255
17501,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0,0.388368,5.805749


## Завдання 7: Провести One Hot Encoding для категоріального атрибута

In [8]:
def apply_one_hot_encoding(dataframe):
    # Візьмемо копію перших 10 000 записів, щоб операція виконувалася миттєво
    sub_df = dataframe.head(10000).copy()
    
    # Створюємо категоріальний стовпець на основі розбиття потужності на 3 рівні
    sub_df['Power_Level'] = pd.qcut(sub_df['Global_active_power'], q=3, labels=['Low', 'Medium', 'High'])
    
    # Виконуємо One Hot Encoding для новоствореного категоріального атрибута
    # Параметр dtype=int перетворить True/False на звичні 1 та 0
    encoded_df = pd.get_dummies(sub_df, columns=['Power_Level'], dtype=int)
    
    return encoded_df

# Викликаємо функцію та виводимо стовпці, що утворилися в кінці таблиці
encoded_result = apply_one_hot_encoding(df_power)
encoded_result[['Global_active_power', 'Power_Level_Low', 'Power_Level_Medium', 'Power_Level_High']].head()

,Global_active_power,Power_Level_Low,Power_Level_Medium,Power_Level_High
0,4.216,0,0,1
1,5.360,0,0,1
2,5.374,0,0,1
3,5.388,0,0,1
4,3.666,0,0,1
